In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import pandas as pd
import matplotlib.pyplot as plt
import re

In [2]:
bbc_data = pd.read_csv('bbc_news.csv')

In [3]:
bbc_data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [4]:
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


Now, it would be intresting to see which parts of speech appear in these article, and what kind of named Entities like ppl, plcaes, organization etc would appear.

Therefore this thing for now will only focus on the title column's NER and POS

In [5]:
titles = pd.DataFrame(bbc_data['title'])

In [6]:
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


The next step is to clean up these titles and i.e, Text_Pre_Processing

In [7]:
#Lower Case
titles['titles_lowercase'] = titles['title'].str.lower()
titles.head(2)

,title,titles_lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...


In [8]:
#Stopwords Removed
en_stopwords = stopwords.words('english')
titles['titles_lowercase_stopwords'] = titles['titles_lowercase'].apply(lambda x: ' '.join([word for word in x.split() if word not in (en_stopwords)]))
titles.head(2)


,title,titles_lowercase,titles_lowercase_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...


In [9]:
#punctations removal
#titles['titles_lowercase_stopwords_punct'] = titles['titles_lowercase_stopwords'].apply(lambda x: re.sub(r"([^\w\s])", "", str(x)))
#titles.head(2)
# just unsafe to convert the coulmns contect into str...so good for only one column

In [10]:
#punctations removal coorect
titles['titles_lowercase_stopwords_punct_coorect'] = titles.apply(lambda x: re.sub(r"([^\w\s])", "", x['titles_lowercase_stopwords']),axis=1)
titles.head(2)

,title,titles_lowercase,titles_lowercase_stopwords,titles_lowercase_stopwords_punct_coorect
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil


In [11]:
#TOKENIZITION
titles['tokens_raw'] = titles.apply(lambda x : word_tokenize(x['title']), axis=1)
titles.head(2)

,title,titles_lowercase,titles_lowercase_stopwords,titles_lowercase_stopwords_punct_coorect,tokens_raw
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,..."


In [12]:
titles['tokens_clean'] = titles.apply(lambda x: word_tokenize(x['titles_lowercase_stopwords_punct_coorect']),axis=1)
titles.head(2)

,title,titles_lowercase,titles_lowercase_stopwords,titles_lowercase_stopwords_punct_coorect,tokens_raw,tokens_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."


In [13]:
# lemmatizartion
lemmatizer = WordNetLemmatizer()
titles['tokens_clean_lemmatized'] = titles['tokens_clean'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
titles.head(2)

,title,titles_lowercase,titles_lowercase_stopwords,titles_lowercase_stopwords_punct_coorect,tokens_raw,tokens_clean,tokens_clean_lemmatized
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic..."


In [27]:
#summarizing the CLEANED AND RAW TOKEN ANDS all
tokens_raw_list = sum(titles['tokens_raw'], [])
tokens_clean_list = sum(titles['tokens_clean_lemmatized'], [])
#print("token_raw_list: ", tokens_raw_list)
#print("tokens_clean_list: ", tokens_clean_listz
tokens_raw_list

['Can',
 'I',
 'refuse',
 'to',
 'work',
 '?',
 "'Liz",
 'Truss',
 'the',
 'Brief',
 '?',
 "'",
 'World',
 'reacts',
 'to',
 'UK',
 'political',
 'turmoil',
 'Rationing',
 'energy',
 'is',
 'nothing',
 'new',
 'for',
 'off-grid',
 'community',
 'The',
 'hunt',
 'for',
 'superyachts',
 'of',
 'sanctioned',
 'Russian',
 'oligarchs',
 'Platinum',
 'Jubilee',
 ':',
 '70',
 'years',
 'of',
 'the',
 'Queen',
 'in',
 '70',
 'seconds',
 'Red',
 'Bull',
 'found',
 'guilty',
 'of',
 'breaking',
 'Formula',
 '1',
 "'s",
 'budget',
 'cap',
 'World',
 'Triathlon',
 'Championship',
 'Series',
 ':',
 'Flora',
 'Duffy',
 'beats',
 'Georgia',
 'Taylor-Brown',
 'to',
 'women',
 "'s",
 'title',
 'Terry',
 'Hall',
 ':',
 'Coventry',
 'scooter',
 'ride-out',
 'pays',
 'tribute',
 'to',
 'singer',
 'Post',
 'Office',
 'and',
 'Fujitsu',
 'to',
 'face',
 'inquiry',
 'over',
 'Horizon',
 'scandal',
 "'Pavement",
 'parking',
 'frightens',
 'me',
 "'",
 'UK',
 'interest',
 'rates',
 ':',
 'How',
 'will',
 'the'

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


POST TAGGING

In [15]:
nlp = spacy.load('en_core_web_sm')

In [16]:
spacy_doc = nlp(' '.join(tokens_raw_list))

In [17]:
pos_df = pd.DataFrame(columns=['token', 'pos_tag'])

for i in spacy_doc:
    pos_df = pd.concat([pos_df, 
                       pd.DataFrame.from_records([{'token': i.text, 'pos_tag': i.pos_}])], ignore_index= True)

In [18]:
# token freqiency count - in other wrods we want to count how many times each word appeard in our document and which part of speech it belongs to 
# to do this lets create a new data frame called pos_def_counts

pos_df_counts = pos_df.groupby(['token', 'pos_tag']).size().reset_index(name='counts').sort_values(by='counts',ascending= False)
pos_df_counts.head(10)
# pos_df.groupby - we group the data bu two columns the token column whihc contains the word itself, and the pos tag column wchihc contains its part of speech
# size - this counts how many rows fall into each of those groups (how okften each token-tag pai occurs)
# reset_index() - we use this, cuz when we grouped the data, pandas automatically placed the gourped cloumns into the index, and the putput is no londer displayed like regualr dataframe with normal cloumns, to fix this reset_index removes those group labels from the index and turns them back into regualr columns.(this gives usclean table again, where token, pos_tag, and count each have theri own column )
# sort_valuers() - so that the most frequent token appear on top

,token,pos_tag,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172
22,-,PUNCT,166
4043,the,DET,163
1856,and,CCONJ,147
15,'s,PART,143
97,?,PUNCT,130


since we ran this on the raw data, the reuslts arent very clean, the ouput includes a lot of punct and siotpwords

In [19]:
#lets get usefull insights focusing on specific parts of speech

In [20]:
nouns = pos_df_counts[pos_df_counts.pos_tag == 'NOUN'][0:10]
nouns

,token,pos_tag,counts
4267,war,NOUN,35
3552,record,NOUN,15
3416,police,NOUN,14
4356,year,NOUN,14
4316,win,NOUN,14
3061,living,NOUN,13
4009,tax,NOUN,13
2326,day,NOUN,12
3368,people,NOUN,12
2031,boss,NOUN,11


In [21]:
verbs =pos_df_counts[pos_df_counts.pos_tag == 'VERB'][0:10]
verbs

,token,pos_tag,counts
3687,says,VERB,30
9,',VERB,14
2670,found,VERB,13
4317,win,VERB,12
4324,wins,VERB,10
2713,get,VERB,9
2388,dies,VERB,9
3990,take,VERB,8
2982,killed,VERB,8
3686,say,VERB,8


In [22]:
adj  =pos_df_counts[pos_df_counts.pos_tag == 'ADJ'][0:10]
adj

,token,pos_tag,counts
3244,new,ADJ,28
1400,Russian,ADJ,21
2606,final,ADJ,16
19,-,ADJ,14
2625,first,ADJ,12
3199,more,ADJ,10
1994,big,ADJ,9
2835,high,ADJ,9
3000,last,ADJ,8
3304,other,ADJ,8


### NOW TRY DOING POS TAGGINF ON TE CLEAN TOKEN AND COMAPRE THE REUSLTS ----- ACTIVUTY

### NER

In [23]:
ner_df = pd.DataFrame(columns=['token', 'ner_tag'])

for i in spacy_doc.ents: # all the entities in our spacy doc ents
    if pd.isna(i.label_) is False: # if the token has a valid label, then pricess it
        ner_df = pd.concat([ner_df, pd.DataFrame.from_records(
            [{'token': i.text, 'ner_tag': i.label_}]
        )], ignore_index=True)
    
# we the take the token and labe_ which give s us the human readealbe version of the named entity tag, we then use from records again to create a one row data frame fromthis info and concat it to ner_df  

# if pd.isna ---> this check if missing vlaues, here it is ued to check missing labels  

In [24]:
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [25]:
#most comman entites in out data set

In [26]:
ner_df_counts = ner_df.groupby(['token', 'ner_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending = False)
ner_df_counts.head()

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
